# Wavebit Tutorial 3
Device - CPU and GPU

## Running the GPU sections

The GPU cells below need a **CUDA** device. They are guarded: without one they print a
notice and compute nothing. That is deliberate — CPU timings printed under a "GPU"
heading would be read as GPU timings, and a misleading number is worse than no number.

### Apple Silicon (Metal / MPS) will not work here

This is not a configuration you can switch on. Wavebit solves in **complex128**
end to end, and Metal supports neither `float64` nor `complex128`:

```
torch.zeros(2, dtype=torch.float32,    device='mps')   # OK
torch.zeros(2, dtype=torch.complex64,  device='mps')   # OK
torch.zeros(2, dtype=torch.float64,    device='mps')   # RuntimeError
torch.zeros(2, dtype=torch.complex128, device='mps')   # RuntimeError
```

That rules out both the PyTorch MPS backend and Apple's experimental `jax-metal`
plugin, which has the same limitation.

Dropping to single precision is not a setting either. The solve is an eigendecomposition
of large non-symmetric complex matrices; in float32 it loses roughly eight significant
digits, which is enough to break agreement between the numpy, jax and torch backends.
Supporting Metal would mean designing a separate single-precision path and deciding what
accuracy it is allowed to claim — a project, not a flag.

So: to fill in the GPU cells, run this notebook on a CUDA machine.

In [ ]:
import warnings
# The 32-bit runs below trip UnresolvedBranchWarning on purpose -- it is the cost this
# tutorial is about. Python's default format prefixes the message with the absolute
# path of the source line, which must not ship in a stored output (docs/README.md, and
# tests/test_docs_contract.py enforces it), so print the category and message only.
warnings.formatwarning = lambda msg, cat, *a, **k: f'{cat.__name__}: {msg}\n'

import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

import time
import numpy as np

import wavebit

import jax

# The GPU cells below are skipped unless this is True. Apple Metal (jax-metal) does
# not count: it has no float64/complex128, which this solver needs throughout.
GPU_AVAILABLE = any(d.platform == 'gpu' for d in jax.devices())
print('GPU available:', GPU_AVAILABLE, '| jax devices:', jax.devices())


In [ ]:
# experiment options
psi = np.pi / 2  # pure s (TE-like); psi is the linear angle

top = 1.0  # superstrate: relative permittivity. 1.0 is vacuum
bot = 1.0  # substrate

theta = 20 * np.pi / 180
phi = 50 * np.pi / 180

wavelength = 900

thickness = [500]
period = dict(x=1000, y=1000)

truncation_order = dict(x=15, y=15)
# truncation_order = dict(x=3, y=3)
sample_x, sample_y, sample_z = 20, 20, 20

structure = np.array([
            [
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
                [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, ],
            ],
        ]) * 24 + 1  # relative permittivity: 1 (vacuum) and 25

## set device

1. at initialization

In [ ]:
backend = 'jax'  # JAX backend
device = 0 # CPU;
dtype = 0
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                     truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure,
                     thickness=thickness, device=device, complex_dtype=dtype)

    

2. after initialization

In [ ]:
backend = 'jax'  # JAX backend

solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                     truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure,
                     thickness=thickness, complex_dtype=dtype)
solver.device = 0

# Test

## JAX

### CPU, 64 bit

In [ ]:
backend = 'jax'  # JAX backend
device = 0 # CPU;
dtype = 0
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                     truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure,
                     thickness=thickness, device=device, complex_dtype=dtype, answers='field')

# JAX dispatches asynchronously, so each timed call is blocked on its result. Without that
# these would be dispatch latency — a cached jit call 'takes' well under a millisecond no
# matter how much work it stands for.
t0 = time.time()
jax.block_until_ready(solver.solve())
print(f'time for efficiency, 1st: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.solve())
print(f'time for efficiency, 2nd: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
print(f'time for field, 1st: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
print(f'time for field, 2nd: ', time.time() - t0)

# solve_field is jit-compiled: it traces the whole solver, so the discrete geometry
# decisions have to be frozen up front. Freeze once and reuse — a fresh freeze is a new
# static argument, so it would recompile and the '2nd' timing below would stop meaning
# 'cached'.
geometry = solver.freeze_geometry()

t0 = time.time()
jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


### GPU, 64 bit

In [ ]:
if not GPU_AVAILABLE:
    print("SKIPPED: no CUDA GPU on this machine.\n"
          "Nothing was computed — see the note at the top of the notebook.\n"
          "Run this notebook on a CUDA machine to fill in these timings.")
else:
    backend = 'jax'  # JAX backend
    device = 1 # GPU;
    dtype = 0
    solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                         truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure,
                         thickness=thickness, device=device, complex_dtype=dtype, answers='field')

    # JAX dispatches asynchronously, so each timed call is blocked on its result. Without that
    # these would be dispatch latency — a cached jit call 'takes' well under a millisecond no
    # matter how much work it stands for.
    t0 = time.time()
    jax.block_until_ready(solver.solve())
    print(f'time for efficiency, 1st: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.solve())
    print(f'time for efficiency, 2nd: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
    print(f'time for field, 1st: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
    print(f'time for field, 2nd: ', time.time() - t0)

    # solve_field is jit-compiled: it traces the whole solver, so the discrete geometry
    # decisions have to be frozen up front. Freeze once and reuse — a fresh freeze is a new
    # static argument, so it would recompile and the '2nd' timing below would stop meaning
    # 'cached'.
    geometry = solver.freeze_geometry()

    t0 = time.time()
    jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
    print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
    print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


### CPU, 32 bit

In [ ]:
backend = 'jax'  # JAX backend
device = 0  # CPU;
dtype = 1  # 32bit
solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                     truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure,
                     thickness=thickness, device=device, complex_dtype=dtype, answers='field')

# JAX dispatches asynchronously, so each timed call is blocked on its result. Without that
# these would be dispatch latency — a cached jit call 'takes' well under a millisecond no
# matter how much work it stands for.
t0 = time.time()
jax.block_until_ready(solver.solve())
print(f'time for efficiency, 1st: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.solve())
print(f'time for efficiency, 2nd: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
print(f'time for field, 1st: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
print(f'time for field, 2nd: ', time.time() - t0)

# solve_field is jit-compiled: it traces the whole solver, so the discrete geometry
# decisions have to be frozen up front. Freeze once and reuse — a fresh freeze is a new
# static argument, so it would recompile and the '2nd' timing below would stop meaning
# 'cached'.
geometry = solver.freeze_geometry()

t0 = time.time()
jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

t0 = time.time()
jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)


### GPU, 32 bit

In [ ]:
if not GPU_AVAILABLE:
    print("SKIPPED: no CUDA GPU on this machine.\n"
          "Nothing was computed — see the note at the top of the notebook.\n"
          "Run this notebook on a CUDA machine to fill in these timings.")
else:
    backend = 'jax'  # JAX backend
    device = 1  # CPU;
    dtype = 1  # 32bit
    solver = wavebit.rcwa(backend=backend, psi=psi, top=top, bot=bot, theta=theta, phi=phi,
                         truncation_order=truncation_order, wavelength=wavelength, period=period, structure=structure,
                         thickness=thickness, device=device, complex_dtype=dtype, answers='field')

    # JAX dispatches asynchronously, so each timed call is blocked on its result. Without that
    # these would be dispatch latency — a cached jit call 'takes' well under a millisecond no
    # matter how much work it stands for.
    t0 = time.time()
    jax.block_until_ready(solver.solve())
    print(f'time for efficiency, 1st: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.solve())
    print(f'time for efficiency, 2nd: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
    print(f'time for field, 1st: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.reconstruct_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z))
    print(f'time for field, 2nd: ', time.time() - t0)

    # solve_field is jit-compiled: it traces the whole solver, so the discrete geometry
    # decisions have to be frozen up front. Freeze once and reuse — a fresh freeze is a new
    # static argument, so it would recompile and the '2nd' timing below would stop meaning
    # 'cached'.
    geometry = solver.freeze_geometry()

    t0 = time.time()
    jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
    print(f'time for efficiency and field in one step, 1st: ', time.time() - t0)

    t0 = time.time()
    jax.block_until_ready(solver.solve_field(sample_x=sample_x, sample_y=sample_y, sample_z=sample_z, geometry=geometry))
    print(f'time for efficiency and field in one step, 2nd: ', time.time() - t0)
